# Migration Phase 1: Behavioral Data (`BehavioralLoader`)

`src/neuroalign/data/loaders/behavioral.py` wraps `brainlink.BrainLinkDB` and
becomes the **canonical session list** for the pipeline (replacing
`SESSIONS_CSV` / `QuestionnaireLoader`).

- `get_sessions()` -> one row per session: `uid`, `subject_code`, `session_id`,
  `AGE` (renamed from `age_at_scan`), `sex`, plus other demographics.
- `get_questionnaires()` -> same, plus pivoted questionnaire columns.


In [1]:
from pathlib import Path
from dotenv import load_dotenv


In [2]:
import os

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.loaders import BehavioralLoader

loader = BehavioralLoader(os.environ["BRAINLINK_DB_PATH"])
print("BehavioralLoader OK")

BehavioralLoader OK


## 1. `get_sessions()`

Defaults: `require_complete_mapping=True` (only sessions with `uid` +
`subject_code` + `session_id` all present).

In [3]:
sessions = loader.get_sessions()
print(f"{sessions.shape[0]} sessions, {sessions.shape[1]} columns")
print(sessions.columns.tolist())
sessions[["uid", "subject_code", "session_id", "lab", "AGE", "sex"]].head()

4938 sessions, 17 columns
['session_id', 'uid', 'subject_code', 'subject_code_bids', 'lab', 'scan_date', 'scan_tag', 'scan_number', 'protocol', 'study', 'group_label', 'mapping_complete', 'AGE', 'sex', 'weight_kg', 'height_m', 'dominant_hand']


,uid,subject_code,session_id,lab,AGE,sex
0,S004192,0336,202012061756,YA,25.42,Female
1,S582702,BAL45,201902171516,YA,29.55,Female
2,S020270,0137,201912111411,YA,25.45,Female
3,S632872,0357,202012301703,YA,30.19,Female
4,S328700,B031,201901021128,YA,29.26,Female


### Age / sex distribution

Sanity-check the demographics columns that downstream `feature_store.save_metadata()`
and modeling code rely on (`AGE`, `sex`).

In [7]:
print(sessions["AGE"].describe())
print()
print(sessions["sex"].value_counts(dropna=False))

count    4934.000000
mean       31.142529
std         9.878566
min         9.234771
25%        24.523614
50%        28.130000
75%        35.030601
max        87.270000
Name: AGE, dtype: float64

sex
Male      2891
Female    2045
None         2
Name: count, dtype: int64


## 2. Filtering by lab / required imaging

`labs` and `require_imaging` pass through to `BrainLinkDB.query()`.

In [8]:
print(sessions["lab"].value_counts())

ya_sessions = loader.get_sessions(labs=["YA"])
print(f"\nYA-only sessions: {len(ya_sessions)}")

lab
YA      3023
SNBB    1295
YBH      267
IT       237
TS        46
MG        22
YG        17
TBS        9
GY         9
YY         6
LM         5
           2
Name: count, dtype: int64

YA-only sessions: 3023


## 3. `get_questionnaires()`

Returns the same session-level table plus pivoted questionnaire item columns.

In [9]:
questionnaires = loader.get_questionnaires()
extra_cols = [c for c in questionnaires.columns if c not in sessions.columns]
print(f"{questionnaires.shape[0]} sessions, {len(extra_cols)} questionnaire columns")
print("example questionnaire columns:", extra_cols[:15])

4938 sessions, 180 questionnaire columns
example questionnaire columns: ['questionnaire_version', 'FirstScanningDate', 'QTimeStamp', 'Scan2Q', 'Privacy_Statement', 'Recontact', 'LAB', 'Gender', 'Age', 'DominantHand', 'Weight_(kg)', 'Height_(cm)', 'Gender_Indentity', 'Sexual_Orientation', 'Country_of_Birth']


## 4. Cross-check against tabular derivatives

Confirm `uid`s from `get_sessions()` line up with `sub-<uid>/` directories under
`TABULAR_DERIVATIVES_ROOT` (same check as Phase 0, now via the loader).

In [ ]:
tabular_root = Path(os.environ["TABULAR_DERIVATIVES_ROOT"])

uids = sessions["uid"].unique()
have_dir = [(tabular_root / f"sub-{uid}").exists() for uid in uids]
n_have_dir = sum(have_dir)
print(f"{n_have_dir} / {len(uids)} uids have a sub-<uid>/ directory under TABULAR_DERIVATIVES_ROOT")